In [5]:
import numpy as np
import pandas as pd

import optuna
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan

/Users/nathabit/Documents/sentiment_analysis_nat_habit/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("artifacts/preprocessed_data.csv")

In [3]:
df.head()

,review,processed_reviews
0,Not lasting Easily brakeble,notlasting easily breakable
1,Breaks easily. Not a sturdy product. Don't buy...,breaks easily nota sturdy product buy used pro...
2,Substandard product Broke after one use. Subst...,substandard product broke one used substandard...
3,Received broken 2 combs were broken. Can I get...,received broken 2 combs broken get refund 2 co...
4,Very hard teeth Very hard teeth. It is damagin...,hard teeth hard teeth damaging scalp


In [4]:
df = df['processed_reviews']

In [6]:
# Objective function for Optuna
def objective(trial):
    # Hyperparameters to tune
    n_neighbors = trial.suggest_int("n_neighbors", 3,100)
    min_dist = trial.suggest_float("min_dist", 0.0, 0.5)
    n_components =trial.suggest_int("n_components",2,10)

    min_cluster_size = trial.suggest_int("min_cluster_size", 5, 50)
    epsilon = trial.suggest_float("epsilon", 0.0, 0.5)
    # Select embedding model as a hyperparameter
    sent_model = trial.suggest_categorical("sent_model", [
        "all-MiniLM-L6-v2",
        "multi-qa-mpnet-base-dot-v1",
        "paraphrase-MiniLM-L6-v2",
        "all-mpnet-base-v2",
        "distiluse-base-multilingual-cased"
    ])

    # Define UMAP and HDBSCAN with suggested hyperparameters
    umap_model = UMAP(n_neighbors=n_neighbors, min_dist=min_dist, n_components=n_components)
    
    hdbscan_model = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, cluster_selection_epsilon=epsilon)

    # Load Sentence Transformer Model
    embedding_model = SentenceTransformer(sent_model)

    # Create BERTopic model
    topic_model = BERTopic(umap_model=umap_model, hdbscan_model=hdbscan_model, embedding_model=embedding_model)

    # Fit Model
    topics, _ = topic_model.fit_transform(df)

    # Metric: Number of unique topics (more balanced topic distribution is better)
    num_topics = len(set(topics)) - (1 if -1 in topics else 0)  # Ignore outliers (-1)

    return num_topics  # Maximize number of detected topics

In [ ]:
# Run Optuna Study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=25)

[I 2025-04-01 19:42:27,979] A new study created in memory with name: no-name-eba9b83e-74e4-428c-9d87-d532d197b43c
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
[I 2025-04-01 19:44:08,836] Trial 0 finished with value: 3.0 and parameters: {'n_neighbors': 90, 'min_dist': 0.12700358282191615, 'n_components': 2, 'min_cluster_size': 40, 'epsilon': 0.3892677855469487, 'sent_model': 'distiluse-base-multilingual-cased'}. Best is trial 0 with value: 3.0.
[I 2025-04-01 19:45:47,720] Trial 1 finished with value: 3.0 and parameters: {'n_neighbors': 71, 'min_dist': 0.016480625779096925, 'n_components': 9, 'min_cluster_size': 44, 'epsilon': 0.09729950241819968, 'sent_model': 'multi-qa-mpnet-base-dot-v1'}. Best is trial 0 with value: 3.0.
[I 2025-04-01 19:47:10,798] Trial 2 finished with value: 4.0 and parameters: {'n_neighbors': 63, 'min_dist': 0.44686661542826445, 'n_components': 5, 'min_cluster_size': 39, 'epsilon': 0.49874162503396885, 'sent_model

In [ ]:
# Best Hyperparameters
print("Best hyperparameters:", study.best_params)